# OasisSpaces — cloud reconstruction on a free GPU

Turns a walkthrough video (or photos) of a space into an editable point cloud,
using Google Colab's free GPU for the heavy lifting — including the **dense**
reconstruction step that laptops without CUDA cannot run.

**Before running:** menu *Runtime → Change runtime type → T4 GPU*.

Then run the cells top to bottom. At the end you download `cloud.ply` and open
it in the OasisSpaces editor.

Capture tips for good results: move slowly *through* the space (don't pivot in
place), keep 60–80% overlap between views, avoid bare walls, mirrors, and
blown-out windows.

In [ ]:
!nvidia-smi -L || echo 'No GPU! Use Runtime -> Change runtime type -> T4 GPU'

### 1. Install COLMAP (CUDA build) + ffmpeg

Takes a few minutes. We pull the CUDA-enabled COLMAP from conda-forge via
micromamba; if that ever fails, the cell falls back to Ubuntu's CPU-only
COLMAP (everything still works, just without the dense step).

In [ ]:
%%bash
apt-get -qq update && apt-get -qq install -y ffmpeg > /dev/null 2>&1
if ! command -v colmap > /dev/null; then
  curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj bin/micromamba
  export CONDA_OVERRIDE_CUDA=12.4
  ./bin/micromamba create -y -q -p /opt/colmap-env -c conda-forge colmap \
    && ln -sf /opt/colmap-env/bin/colmap /usr/local/bin/colmap \
    || apt-get -qq install -y colmap > /dev/null 2>&1
fi
colmap -h 2>&1 | head -1

### 2. Get the OasisSpaces pipeline

In [ ]:
!pip -q install 'numpy>=2.3'
%cd /content
!rm -rf /content/OasisSpaces && git clone https://github.com/sanskargupta1808/OasisSpaces.git /content/OasisSpaces
%cd /content/OasisSpaces

### 3. Upload your capture

Upload one walkthrough **video**, or a set of overlapping **photos**.
(For big files it is faster to put them in Google Drive, mount it with the
folder icon on the left, and set `source` to the file's Drive path instead.)

In [ ]:
import shutil
from pathlib import Path
from google.colab import files

VIDEO_EXTENSIONS = {'.mp4', '.mov', '.avi', '.mkv', '.webm'}
capture_dir = Path('/content/OasisSpaces/captures')
shutil.rmtree(capture_dir, ignore_errors=True)  # no stale uploads
capture_dir.mkdir(parents=True)
uploaded = files.upload()
for name, data in uploaded.items():
    (capture_dir / name).write_bytes(data)
videos = [n for n in uploaded if Path(n).suffix.lower() in VIDEO_EXTENSIONS]
source = str(capture_dir / videos[-1]) if videos else str(capture_dir)
print('source =', source)

### 4. Reconstruct

Sparse reconstruction takes minutes; with a CUDA COLMAP the `--dense` step
then builds a far denser cloud (this is the slow part — up to an hour for
long captures). Remove `--dense` if you only want the quick sparse cloud.

In [ ]:
!python3 pipeline/reconstruct.py "{source}" --name colab-space --fps 4 --dense

### 5. Quick preview

In [ ]:
import sys
sys.path.insert(0, 'pipeline')
import numpy as np
import matplotlib.pyplot as plt
from pointcloud import load_ply

cloud = load_ply('spaces/colab-space/cloud.ply')
keep = np.random.default_rng(0).choice(
    len(cloud), min(len(cloud), 60_000), replace=False)
pts, cols = cloud.points[keep], cloud.colors[keep] / 255
fig = plt.figure(figsize=(9, 9))
ax = fig.add_subplot(projection='3d')
ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2], c=cols, s=0.5)
ax.set_axis_off()
ax.set_title(f'{len(cloud):,} points')
plt.show()

### 6. Download and edit

Download the cloud, then open it in the OasisSpaces editor (the hosted one,
or `editor/index.html` from the repo served locally) — drag the file in,
box-select, delete, crop, export.

In [ ]:
from google.colab import files
files.download('spaces/colab-space/cloud.ply')

### Going further: Gaussian splatting

The COLMAP workspace this run produced (`spaces/colab-space/workspace`) —
solved camera poses + images — is exactly the input Gaussian-splatting
trainers (nerfstudio / gsplat / OpenSplat) need for photorealistic results.
Zip it with the cell below if you want to keep it for that.


In [ ]:
!zip -qr colab-space-workspace.zip spaces/colab-space/workspace
from google.colab import files
files.download('colab-space-workspace.zip')